# Build the feature CSV → save it to Google Drive

Clones **[GazeVLM-HWSW-Codesign](https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign)**,
runs the repo's own script on **2 videos**, and saves the result to Drive so it outlives
the Colab session.

```bash
python -m src.dataprep.build_feature_csv --n_videos 2 --cleanup_raw
```

## Columns

| Column | Per row | What |
|---|---|---|
| `idx`, `sequence` | 1 | global row index, video name |
| `feat_frame_1`, `feat_frame_2` | 1 | paths to the frozen DINOv2 `.npz` |
| `frame_similarity` | 1 | CLS cosine — teacher label |
| `gaze_patch_token_sim` | 1 | gaze-patch cosine — teacher label |
| `n_velocities` | 1 | steps in the window (**9**) |
| `gaze_rates_window` | **9 × 3** | ω_yaw, ω_pitch, ω_mag per step |
| `n_gaze_in_gap` | 1 | raw gaze samples in the window (**10**) |
| `n_oof_in_gap` | 1 | how many of them fell outside the lens |
| `gaze_xy_window` | **10 × 2** | projected image coords, normalised |
| `gaze_vec3d_window` | **10 × 3** | unit gaze direction on the sphere |

The last four are new. Note the shapes differ by one: **10 samples give 9 velocities**,
so the rates are the differences *between* the points in `gaze_xy_window`.

## The CSV alone is not enough

`feat_frame_1` and `feat_frame_2` are **paths** to `.npz` feature files. Saving only the
CSV would leave those paths pointing at `/content/...`, which disappears when the runtime
ends — the file would look fine and be useless.

So this notebook copies **both**, and rewrites the paths inside the CSV to their Drive
locations. The result is self-contained: reload it in any future session and it works.

| Saved to Drive | What it is |
|---|---|
| `feature_dataset.csv` | the table, with Drive paths |
| `features/<sequence>/feat_*.npz` | the frozen DINOv2 features |

## Cost

2 videos ≈ **5 GB** downloaded (deleted afterwards by `--cleanup_raw`), ~13 MB kept.
Expect ~4 min. **No GPU needed.**

## 1 — Clone the repo and install

In [ ]:
!git clone -q https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git /content/GazeVLM
%cd /content/GazeVLM
!git log --oneline -1

!pip -q install -r requirements.txt
!pip -q install projectaria-tools

import os, glob, json, shutil, time
import numpy as np, pandas as pd
print("\nrepo:", os.getcwd())

## 2 — Mount Drive and pick where things go

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/GazeVLM"        # <- change if you like
DRIVE_FEATS = os.path.join(DRIVE_DIR, "features")
DRIVE_CSV   = os.path.join(DRIVE_DIR, "feature_dataset.csv")
os.makedirs(DRIVE_FEATS, exist_ok=True)

print("will save to:")
print("   ", DRIVE_CSV)
print("   ", DRIVE_FEATS + "/<sequence>/feat_*.npz")

## 3 — The download-links JSON

If you already keep it in Drive, use Option B and skip the upload every session.

In [ ]:
# ---- Option A: upload from your laptop ----
from google.colab import files
up = files.upload()                       # pick your aea_download_urls.json
URLS_JSON = "/content/" + list(up.keys())[0]
os.rename(list(up.keys())[0], URLS_JSON)

# ---- Option B: already in Drive ----
# URLS_JSON = os.path.join(DRIVE_DIR, "aea_download_urls.json")

# keep a copy in Drive so future sessions can use Option B
kept = os.path.join(DRIVE_DIR, "aea_download_urls.json")
if os.path.abspath(URLS_JSON) != os.path.abspath(kept):
    shutil.copy2(URLS_JSON, kept)
    print("copied the JSON to Drive for next time:", kept)

print(f"{len(json.load(open(URLS_JSON))['sequences'])} videos available")
print("NOTE: the links expire after ~14 days -- re-download the JSON when builds fail.")

## 4 — Build, on local disk

Built in `/content` rather than straight to Drive: writing hundreds of small `.npz` files
over the Drive mount is far slower. They get copied across in one go afterwards.

Every video is used in full (~190 rows each at 1 FPS), so expect **~380 rows**.

The per-sequence table printed at the end now reports `gaze_samples`, `samples_per_row`
and `oof_samples` — check that `samples_per_row` is ≈ 10.0 and `oof_samples` is small.

In [ ]:
N_VIDEOS  = 2
SEED      = 0
LOCAL_CSV = "/content/data/feature_dataset.csv"
LOCAL_FEAT = "/content/data/features"

t0 = time.time()
!python -m src.dataprep.build_feature_csv \
    --urls_json  "{URLS_JSON}" \
    --out_csv    "{LOCAL_CSV}" \
    --raw_dir    /content/data/raw \
    --frames_dir /content/data/frames_1fps \
    --feat_dir   "{LOCAL_FEAT}" \
    --n_videos   {N_VIDEOS} \
    --seed       {SEED} \
    --cleanup_raw

print(f"\nbuild took {(time.time()-t0)/60:.1f} min")

## 5 — Copy to Drive and rewrite the paths

This is the step that makes the saved CSV actually reusable.

In [ ]:
# --- features ---------------------------------------------------------------
t0 = time.time()
n_files = 0
for seq in sorted(os.listdir(LOCAL_FEAT)):
    src_dir = os.path.join(LOCAL_FEAT, seq)
    dst_dir = os.path.join(DRIVE_FEATS, seq)
    if not os.path.isdir(src_dir):
        continue
    os.makedirs(dst_dir, exist_ok=True)
    for f in sorted(os.listdir(src_dir)):
        shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))
        n_files += 1
    print(f"   {seq}: {len(os.listdir(dst_dir))} files")
print(f"copied {n_files} feature files in {time.time()-t0:.0f}s")

# --- CSV, with paths pointing at Drive --------------------------------------
df = pd.read_csv(LOCAL_CSV)

def to_drive(p):
    """/content/data/features/<seq>/feat_x.npz -> <DRIVE_FEATS>/<seq>/feat_x.npz
    Keeps the <sequence>/<file> tail: the basename alone is NOT unique, since every
    sequence folder contains a feat_00000.npz."""
    parts = str(p).replace("\\", "/").rstrip("/").split("/")
    return os.path.join(DRIVE_FEATS, *parts[-2:])

for col in ("feat_frame_1", "feat_frame_2"):
    df[col] = df[col].apply(to_drive)

df.to_csv(DRIVE_CSV, index=False)
print(f"\nwrote {len(df)} rows x {df.shape[1]} columns -> {DRIVE_CSV}")
print(f"   {os.path.getsize(DRIVE_CSV)/1e3:.0f} KB")
print(f"\ncolumns: {list(df.columns)}")
print(f"\nexample rewritten path:\n   {df.loc[0, 'feat_frame_1']}")

## 6 — Verify what landed in Drive

Reads the CSV **back from Drive**, opens the feature files it points at, and recomputes a
similarity from them. If this passes, the saved dataset is self-contained.

In [ ]:
chk = pd.read_csv(DRIVE_CSV)
print(f"{len(chk)} rows x {chk.shape[1]} columns from {chk['sequence'].nunique()} videos")
display(chk.groupby("sequence").size().rename("rows").to_frame())

missing = [p for p in pd.concat([chk.feat_frame_1, chk.feat_frame_2]).unique()
           if not os.path.exists(p)]
print(f"\nreferenced feature files missing: {len(missing)}")

z0 = np.load(chk.loc[0, "feat_frame_1"])
z1 = np.load(chk.loc[0, "feat_frame_2"])
fs = float(np.dot(z0["cls"], z1["cls"]))            # tokens are L2-normalised

g = int(z0["grid"])
def cell(z):
    x, y = z["gaze_xy"]
    gx = min(g-1, int(np.clip(x, 0, 1)*g)); gy = min(g-1, int(np.clip(y, 0, 1)*g))
    return z["patches"][gy*g + gx]
ps = float(np.dot(cell(z0), cell(z1)))

ok = (len(missing) == 0
      and abs(fs - chk.loc[0, "frame_similarity"]) < 1e-3
      and abs(ps - chk.loc[0, "gaze_patch_token_sim"]) < 1e-3)
print(f"\nrecomputed from Drive vs the CSV:")
print(f"   frame_similarity     {fs:.4f}  vs  {chk.loc[0,'frame_similarity']:.4f}")
print(f"   gaze_patch_token_sim {ps:.4f}  vs  {chk.loc[0,'gaze_patch_token_sim']:.4f}")
print(f"\n   [{'PASS' if ok else 'FAIL'}]  the dataset in Drive is self-contained")

sz = sum(os.path.getsize(os.path.join(r, f))
         for r, _, fs_ in os.walk(DRIVE_FEATS) for f in fs_) / 1e6
print(f"\nDrive usage: {sz:.1f} MB of features + {os.path.getsize(DRIVE_CSV)/1e3:.0f} KB CSV")
display(chk.head(5))

## 7 — The two new per-sample columns

`gaze_xy_window` and `gaze_vec3d_window` hold **one entry per raw gaze sample**, so ~10
per 1 s window — one more than `gaze_rates_window`, which holds the 9 differences
*between* them.

Both unpack with the repo's own `parse_rates`, which splits on `;` then `,` and therefore
handles the 2-wide and 3-wide forms identically.

In [ ]:
import sys; sys.path.insert(0, "/content/GazeVLM")
from src.loss1.dataset import parse_rates

ROW = 0
r = chk.loc[ROW]

XY   = parse_rates(r["gaze_xy_window"])       # (n, 2)
VEC  = parse_rates(r["gaze_vec3d_window"])    # (n, 3)
RATE = parse_rates(r["gaze_rates_window"])    # (n-1, 3)

print(f"row {ROW}   sequence {r['sequence']}")
print(f"   n_gaze_in_gap {r['n_gaze_in_gap']}   n_oof_in_gap {r['n_oof_in_gap']}   "
      f"n_velocities {r['n_velocities']}\n")
print(f"   gaze_xy_window     -> {XY.shape}")
print(f"   gaze_vec3d_window  -> {VEC.shape}")
print(f"   gaze_rates_window  -> {RATE.shape}   <- one FEWER, they are the differences\n")

print("   sample |      x        y    |     vx        vy        vz")
print("   -------+--------------------+------------------------------")
for k in range(len(XY)):
    print(f"   {k:6d} | {XY[k,0]:8.4f} {XY[k,1]:8.4f} | "
          f"{VEC[k,0]:+9.5f} {VEC[k,1]:+9.5f} {VEC[k,2]:+9.5f}")

### Checks

Five things worth asserting before this data is trusted downstream.

In [ ]:
XYs  = [parse_rates(s) for s in chk["gaze_xy_window"]]
VECs = [parse_rates(s) for s in chk["gaze_vec3d_window"]]

n_xy  = np.array([len(a) for a in XYs])
n_vec = np.array([len(a) for a in VECs])
n_rat = chk["n_velocities"].to_numpy()

allxy  = np.concatenate([a for a in XYs if len(a)])
allvec = np.concatenate([a for a in VECs if len(a)])
norms  = np.linalg.norm(allvec, axis=1)

checks = [
    ("xy and vec3d have the same length",
     bool((n_xy == n_vec).all()),
     f"{int((n_xy != n_vec).sum())} rows disagree"),

    ("length matches n_gaze_in_gap",
     bool((n_xy == chk['n_gaze_in_gap'].to_numpy()).all()),
     f"median {int(np.median(n_xy))} samples per row"),

    ("velocities == samples - 1",
     bool((n_rat == n_xy - 1).all()),
     f"{int(np.median(n_rat))} velocities from {int(np.median(n_xy))} samples"),

    ("vec3d rows are UNIT vectors",
     bool(np.allclose(norms, 1.0, atol=1e-5)),
     f"norm range [{norms.min():.8f}, {norms.max():.8f}]"),

    ("xy inside the image",
     bool((allxy >= 0).all() and (allxy <= 1).all()),
     f"x [{allxy[:,0].min():.3f}, {allxy[:,0].max():.3f}]  "
     f"y [{allxy[:,1].min():.3f}, {allxy[:,1].max():.3f}]"),
]

print(f"{len(chk)} rows, {len(allxy):,} gaze samples total\n")
for name, okk, detail in checks:
    print(f"   [{'PASS' if okk else 'FAIL'}]  {name:36s} {detail}")

oof = int(chk["n_oof_in_gap"].sum())
print(f"\n   out-of-FOV samples: {oof:,} / {len(allxy):,} ({100*oof/len(allxy):.2f}%)")
print("   Those carry an INVENTED (0.5, 0.5) in gaze_xy_window and are indistinguishable")
print("   from a real centre gaze -- filter on n_oof_in_gap before using the xy column.")
print("   gaze_vec3d_window is unaffected: it needs no camera model.")

### What the two columns look like

Left: where gaze landed on the image. Right: the same motion as directions on the
sphere — no camera model involved, so this one is immune to any projection error.

In [ ]:
import matplotlib.pyplot as plt

SEQ = chk["sequence"].iloc[0]
sub = chk[chk["sequence"] == SEQ]
XYs_s  = np.concatenate([parse_rates(s) for s in sub["gaze_xy_window"]])
VECs_s = np.concatenate([parse_rates(s) for s in sub["gaze_vec3d_window"]])

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))

ax[0].scatter(XYs_s[:, 0], XYs_s[:, 1], s=3, alpha=.25)
ax[0].set_xlim(0, 1); ax[0].set_ylim(1, 0)      # image rows grow downward
ax[0].set_xlabel("x"); ax[0].set_ylabel("y")
ax[0].set_title(f"gaze_xy_window — {len(XYs_s):,} samples")

ax[1].scatter(VECs_s[:, 0], VECs_s[:, 1], s=3, alpha=.25, c="tab:orange")
ax[1].axhline(0, lw=.4, c="k"); ax[1].axvline(0, lw=.4, c="k")
ax[1].set_xlabel("vx  (+ = right)"); ax[1].set_ylabel("vy  (+ = up)")
ax[1].set_aspect("equal"); ax[1].set_title("gaze_vec3d_window (x-y plane)")

one = parse_rates(sub["gaze_xy_window"].iloc[len(sub)//2])
ax[2].plot(one[:, 0], one[:, 1], "-o", ms=6)
ax[2].scatter([one[0, 0]], [one[0, 1]], s=130, c="lime", ec="k", zorder=5, label="first")
ax[2].set_xlim(0, 1); ax[2].set_ylim(1, 0); ax[2].legend()
ax[2].set_title(f"one window: {len(one)} samples of eye motion")

plt.tight_layout(); plt.show()

print("Left  : should cluster where the wearer looked, NOT hug the borders.")
print("Middle: a cloud around the origin -- vz is near 1, so x-y shows the whole spread.")
print("Right : one second of eye movement; short hops are fixation drift, long ones saccades.")

---

## Using it in a later session

```python
from google.colab import drive; drive.mount("/content/drive")
CSV = "/content/drive/MyDrive/GazeVLM/feature_dataset.csv"
```

```bash
python -m src.loss1.train --csv $CSV --out_dir runs/loss1
python -m src.loss2.train --csv $CSV --out_dir runs/loss2
```

No rebuild, no re-download, no DINOv2 — the paths already point into Drive.

Training reads columns **by name**, so the four new ones are simply ignored by
`Loss1Dataset` and `Loss2Dataset`. Nothing downstream needs changing to consume this CSV.

If you ever move the `features/` folder, pass `--feat_root <new location>` instead of
rewriting the CSV; the loaders re-root on the `<sequence>/<file>` tail.

## Adding more videos later

Change `SEED` (or pass `--seqs` with explicit names), rerun, and **write to a different
`--out_csv`**, then concatenate:

```python
big = pd.concat([pd.read_csv(a), pd.read_csv(b)], ignore_index=True)
big["idx"] = range(len(big))          # idx must stay unique and contiguous
big.to_csv("/content/drive/MyDrive/GazeVLM/feature_dataset.csv", index=False)
```

Feature folders are named per sequence, so they merge without collisions.

**An older CSV cannot be merged with a new one** — it has 8 columns, this has 12, and
`pd.concat` would fill the missing four with `NaN` rather than erroring. Rebuild instead:
the new columns cannot be backfilled without the VRS (~2.5 GB per video), because
projecting gaze to `(x, y)` needs the device calibration.

## A note on scale

2 videos gives ~380 rows and a 1 train / 1 val split — enough to check the plumbing, not
enough to train anything meaningful. The models carry ~1.5 M parameters. Build up to tens
of videos in Drive before drawing conclusions from any training run.